In [31]:
#NAIVE RAG using Llama Index

# %pip install llama-index llama-index-llms-ollama llama-index-embeddings-ollama
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

In [32]:
#configuration
Settings.llm = Ollama(model="gemma4:e4b",
                    temperature=0.7,
                    num_predictions=300,
                    request_timeout=600)
Settings.embed_model = OllamaEmbedding(model_name="qwen3-embedding:0.6b")
Settings.chunk_size_limit = 512
Settings.chunk_overlap = 50

In [33]:
#document loading

from llama_index.core import SimpleDirectoryReader
documents = SimpleDirectoryReader("data/").load_data()

In [34]:
#main

from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import VectorStoreIndex

# 1. Parse into nodes explicitly
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(documents)

# 2. Build the index from nodes (not from raw documents)
index = VectorStoreIndex(nodes)

# 3. Build retriever explicitly (this is the piece that changes across architectures)
retriever = index.as_retriever(similarity_top_k=5)

# 4. Build query engine from the retriever
from llama_index.core.query_engine import RetrieverQueryEngine
query_engine = RetrieverQueryEngine.from_args(retriever)

In [36]:
#Query

response = query_engine.query("summerize the introduction")
print(response.response)
print([n.text for n in response.source_nodes])  # this is your `contexts` field for RAGAS/DeepEval

Modern information-retrieval research has evolved significantly, moving from early probabilistic models to neural ranking systems. Within this field, Retrieval-Augmented Generation (RAG) emerged as a key advancement, which pairs a large language model with an external retriever during inference. While the current state-of-the-art utilizes techniques like Dense Passage Retrieval for open-domain question answering, prior comparative studies have methodological shortcomings. These previous evaluations often fail to separate retrieval quality from generation quality, may not specify critical hyperparameters, and report overall scores without systematic error analysis or uncertainty quantification.

This paper addresses these gaps by offering a principled, model-agnostic approach to evaluating RAG. The proposed framework is designed to be both replicable and relevant for deployment. The primary contribution is methodological, aiming to link specific index choices and tuning parameters to me